In [1]:
import torch
import torch.nn as nn
import torch.optim as optim

import pickle
import librosa
import numpy as np
from constants import *
from pathlib import Path
import matplotlib.pyplot as plt
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split

In [2]:
ney_feature_dirs = sorted(
    [f for f in Path(NEY_FEATURE_DIR).iterdir() if f.is_dir()])
gtr_feature_dirs = sorted(
    [f for f in Path(GTR_FEATURE_DIR).iterdir() if f.is_dir()])

In [3]:
x_train_dirs, x_test_dirs, y_train_dirs, y_test_dirs = train_test_split(
    gtr_feature_dirs, ney_feature_dirs, test_size=0.2, random_state=42)

print(len(x_train_dirs), len(x_test_dirs))

20 5


In [4]:
print(x_train_dirs[0], y_train_dirs[0])
print(x_test_dirs[0], y_test_dirs[0])

dataset/features/ac_gtr/09_Gtr_A_3 dataset/features/ney/09_Ney_A_3
dataset/features/ac_gtr/08_Gtr_G_S_3 dataset/features/ney/08_Ney_G_S_3


In [29]:
class SpectrogramDataset(Dataset):
    def __init__(self, x_dirs, y_dirs, feature="real"):
        self.x_dirs = x_dirs
        self.y_dirs = y_dirs
        self.feature = feature
        self.x = None
        self.y = None

        # gtr files
        self.x = self._build_spectrogram_data(self.x_dirs)

        # ney files
        self.y = self._build_spectrogram_data(self.y_dirs)

    def _build_spectrogram_data(self, dir_paths):
        arr = []
        for dir_path in dir_paths:
            arr.extend(self._get_spectrograms_in_dir(dir_path))

        arr = np.array(arr, dtype=np.float32)
        # min max scaling
        arr = (arr - np.min(arr)) / (np.max(arr) - np.min(arr))
        arr = np.expand_dims(arr, axis=1)
        return arr

    def _get_spectrograms_in_dir(self, dir_path):
        spectrograms = []
        sorted_files = sorted(list(Path(dir_path).iterdir()),
                              key=lambda x: int(x.stem.split("_")[1]))
        for f in sorted_files:
            with open(f, "rb") as handle:
                chunk = pickle.load(handle)
                if self.feature == "real":
                    data = chunk["data"][0]
                elif self.feature == "imag":
                    data = chunk["data"][1]
                else:
                    raise Exception(
                        "feature parameter must be either 'real' or 'imag'!")
                spectrograms.append(data)
        return spectrograms

    def __getitem__(self, index):
        return self.x[index], self.y[index]

    def __len__(self):
        return self.x.shape[0]

In [30]:
train_dataset = SpectrogramDataset(x_train_dirs, y_train_dirs, feature="real")
test_dataset = SpectrogramDataset(x_test_dirs, y_test_dirs, feature="real")

In [31]:
train_data_loader = DataLoader(
    dataset=train_dataset,
    batch_size=16,
    shuffle=True)

test_data_loader = DataLoader(
    dataset=test_dataset,
    batch_size=16,
    shuffle=False)

In [33]:
x, y = next(iter(train_data_loader))
print("x[0]:", x[0])
print("Original shape:", x.shape)
# encoder
print("Encoder:")
x = nn.Conv2d(1, 16, 3, stride=1)(x)
print("1", x.shape)
x = nn.Conv2d(16, 32, 3, stride=2)(x)
print("2", x.shape)
x = nn.Conv2d(32, 64, 3, stride=2)(x)
print("3", x.shape)
x = nn.Conv2d(64, 128, 3, stride=1)(x)
print("4", x.shape)
# decoder
print("Decoder:")
x = nn.ConvTranspose2d(128, 64, 3, stride=1)(x)
print("5", x.shape)
x = nn.ConvTranspose2d(64, 32, 3, stride=2)(x)
print("6", x.shape)
x = nn.ConvTranspose2d(32, 16, 5, stride=2)(x)
print("7", x.shape)
x = nn.ConvTranspose2d(16, 1, 4, stride=1)(x)
print("8", x.shape)

x[0]: tensor([[[0.4889, 0.4885, 0.4888,  ..., 0.4898, 0.4901, 0.4903],
         [0.4955, 0.4955, 0.4950,  ..., 0.4943, 0.4942, 0.4933],
         [0.4880, 0.4857, 0.4893,  ..., 0.4877, 0.4896, 0.4929],
         ...,
         [0.4919, 0.4920, 0.4921,  ..., 0.4920, 0.4921, 0.4920],
         [0.4921, 0.4921, 0.4920,  ..., 0.4919, 0.4919, 0.4919],
         [0.4919, 0.4919, 0.4919,  ..., 0.4921, 0.4921, 0.4921]]])
Original shape: torch.Size([16, 1, 512, 152])
Encoder:
1 torch.Size([16, 16, 510, 150])
2 torch.Size([16, 32, 254, 74])
3 torch.Size([16, 64, 126, 36])
4 torch.Size([16, 128, 124, 34])
Decoder:
5 torch.Size([16, 64, 126, 36])
6 torch.Size([16, 32, 253, 73])
7 torch.Size([16, 16, 509, 149])
8 torch.Size([16, 1, 512, 152])


In [26]:
class Gtr_2_Ney_Encoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Conv2d(1, 16, 3, stride=1),
            nn.ReLU(),
            nn.Conv2d(16, 32, 3, stride=2),
            nn.ReLU(),
            nn.Conv2d(32, 64, 3, stride=2),
            nn.ReLU(),
            nn.Conv2d(64, 128, 3, stride=1)
        )

        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(128, 64, 3, stride=1),
            nn.ReLU(),
            nn.ConvTranspose2d(64, 32, 3, stride=2),
            nn.ReLU(),
            nn.ConvTranspose2d(32, 16, 5, stride=2),
            nn.ReLU(),
            nn.ConvTranspose2d(16, 1, 4, stride=1),
            nn.Sigmoid()
        )

    def forward(self, x):
        encoded = self.encoder(x)
        decoded = self.decoder(encoded)
        return decoded

In [27]:
gtr_2_ney_model = Gtr_2_Ney_Encoder()
criterion = nn.L1Loss()
optimizer = optim.Adam(gtr_2_ney_model.parameters(), lr=0.001)

for epoch in range(10):
    for gtr_spectrograms, ney_spectrograms in train_data_loader:
        y_hat = gtr_2_ney_model(gtr_spectrograms)
        loss = criterion(y_hat, ney_spectrograms)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    print(f"Epoch: {epoch + 1}, Loss: {loss.item():.4f}")

Epoch: 1, Loss: 0.0002
Epoch: 2, Loss: 0.0002


KeyboardInterrupt: 

In [73]:
gtr_spectrograms_test, ney_spectrograms_test = next(iter(test_data_loader))
with torch.no_grad():
    predictions = gtr_2_ney_model(gtr_spectrograms_test)

print(predictions.shape)

torch.Size([16, 1, 40, 256])


In [74]:
def restore_predictions(predictions):
    restored_predictions = predictions.numpy()
    restored_predictions = np.transpose(restored_predictions, (0, 3, 2, 1))
    restored_predictions = np.squeeze(restored_predictions)
    return restored_predictions

In [75]:
restored_predictions = restore_predictions(predictions)

In [76]:
def plot_spectrogram(Y, sr, hop_length, y_axis="linear"):
    plt.figure(figsize=(12, 6))
    librosa.display.specshow(
        Y,
        sr=sr,
        hop_length=hop_length,
        x_axis="time",
        y_axis=y_axis)

    plt.colorbar()